In [33]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

In [51]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

df = pd.read_csv("cardio_cleaned.csv", sep=";")

df_raw = df.copy(deep=True)

df_clean = df_raw.copy(deep=True)

In [35]:
df_clean['age_years'] = (df_clean['age'] / 365.25).round(1)

print(df_clean['age_years'].describe())

count   70000.00
mean       53.30
std         6.76
min        29.60
25%        48.40
50%        53.90
75%        58.40
max        64.90
Name: age_years, dtype: float64


In [36]:
df_clean = df_clean.drop(columns=['id'])

Anomalous height and weight values are converted to NaN to preserve the rows, and will be imputed with the Median during model building.

In [37]:
df_clean.loc[(df['height'] < 100) | (df_clean['height'] > 220), 'height'] = np.nan
df_clean.loc[df['weight'] < 30, 'weight'] = np.nan

print("Missing heights after fill:", df_clean['height'].isnull().sum())
print("Missing weights after fill:", df_clean['weight'].isnull().sum())

Missing heights after fill: 30
Missing weights after fill: 7


In [38]:
invalid_bp = (
    (df_clean['ap_hi'] <= 0) |
    (df_clean['ap_hi'] > 300) |
    (df_clean['ap_lo'] <= 0) |
    (df_clean['ap_lo'] > 200) |
    (df_clean['ap_hi'] <= df_clean['ap_lo'])
)

print("Rows to review:", invalid_bp.sum())
print("Total rows:", len(df_clean))
print("Rows remaining if removed:", len(df_clean) - invalid_bp.sum())

Rows to review: 1291
Total rows: 70000
Rows remaining if removed: 68709


In [39]:
print("Systolic BP <= 0:", (df_clean['ap_hi'] <= 0).sum())
print("Systolic BP > 300:", (df_clean['ap_hi'] > 300).sum())

print("Diastolic BP <= 0:", (df_clean['ap_lo'] <= 0).sum())
print("Diastolic BP > 200:", (df_clean['ap_lo'] > 200).sum())

print(
    "Systolic BP <= Diastolic BP:",
    (df_clean['ap_hi'] <= df_clean['ap_lo']).sum()
)

Systolic BP <= 0: 7
Systolic BP > 300: 40
Diastolic BP <= 0: 22
Diastolic BP > 200: 953
Systolic BP <= Diastolic BP: 1236


In [43]:
print("Highest systolic BP:")
print(df_clean.nlargest(10, 'ap_hi')[['ap_hi', 'ap_lo', 'cardio']])

Highest systolic BP:
       ap_hi  ap_lo  cardio
40852  16020     80       1
25464  14020     80       1
25519  14020     80       1
46912  14020     90       1
47253  14020     90       0
55459  13010     80       1
55847  13010     80       0
7763   11500     90       1
51438  11020     80       1
69370   2000    100       1


In [44]:
print("\nHighest diastolic BP:")
print(df_clean.nlargest(10, 'ap_lo')[['ap_hi', 'ap_lo', 'cardio']])


Highest diastolic BP:
       ap_hi  ap_lo  cardio
43326    200  11000       1
2381     140  10000       1
23849    150  10000       1
68538    160  10000       1
43434    130   9800       0
6653     130   9100       1
12086    130   9011       1
32920    150   9011       1
15990    120   8500       0
44042    120   8200       0


In [45]:
print("\nSystolic BP <= Diastolic BP:")
print(
    df_clean.loc[
        df_clean['ap_hi'] <= df_clean['ap_lo'],
        ['ap_hi', 'ap_lo', 'cardio']
    ].head(10)
)


Systolic BP <= Diastolic BP:
     ap_hi  ap_lo  cardio
228    160   1100       1
241    160   1000       1
260    140    800       1
329    160   1000       1
345    140   1000       1
473    150   1033       1
474    120    150       0
559    200   1000       1
567     14     90       1
613    140   1000       1


In [46]:
invalid_bp = (
    (df_clean['ap_hi'] < 70) |
    (df_clean['ap_hi'] > 250) |
    (df_clean['ap_lo'] < 40) |
    (df_clean['ap_lo'] > 150) |
    (df_clean['ap_hi'] <= df_clean['ap_lo'])
)

print("Rows to remove:", invalid_bp.sum())
print("Rows remaining:", len(df_clean) - invalid_bp.sum())

Rows to remove: 1334
Rows remaining: 68666


In [47]:
df_clean = df_clean.loc[~invalid_bp].copy()

print("Shape after cleaning:", df_clean.shape)
print("Remaining invalid BP rows:",
      (
          (df_clean['ap_hi'] < 70) |
          (df_clean['ap_hi'] > 250) |
          (df_clean['ap_lo'] < 40) |
          (df_clean['ap_lo'] > 150) |
          (df_clean['ap_hi'] <= df_clean['ap_lo'])
      ).sum())

Shape after cleaning: (68666, 13)
Remaining invalid BP rows: 0


In [1]:
import pandas as pd
import shutil
from pathlib import Path


file_path = Path("cardio_cleaned.csv")

backup_path = file_path.with_name("cardio_cleaned_backup.csv")
shutil.copy2(file_path, backup_path)

cleaned_df = pd.read_csv(file_path)

cleaned_df["bmi"] = (
    cleaned_df["weight"] / (cleaned_df["height"] / 100) ** 2
).round(2)

cleaned_df.to_csv(file_path, index=False)

print("Updated shape:", cleaned_df.shape)
print("BMI missing values:", cleaned_df["bmi"].isna().sum())

display(cleaned_df[["height", "weight", "bmi"]].head())

Updated shape: (68666, 14)
BMI missing values: 33


,height,weight,bmi
0,168.0,62.0,21.97
1,156.0,85.0,34.93
2,165.0,64.0,23.51
3,169.0,82.0,28.71
4,156.0,56.0,23.01


In [50]:
df_clean.to_csv("cardio_cleaned.csv", index=False)